In [1]:
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')

import pandas as pd
from pathlib import Path
from nighthawk.data import Constraint

CSV_PATH  = Path('/var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv')
MARKET    = 'SPP'
THRESHOLD = -900
META_COLS = ['location', 'physical_condition', 'outage_name',
             'comment on this constraint', 'start_date', 'end_date',
             'wind', 'reserve_zone']
COL_ORDER = ['bid_date', 'monitored', 'DA_mvalue', 'RT_mvalue',
             'location', 'physical_condition', 'outage_name',
              'start_date', 'end_date',
             'wind', 'load', 'reserve_zone', "today's wind", 'opportunity']


def _fetch_mvalues(dt_str: str) -> pd.DataFrame:
    opex  = MARKET
    mv_rt = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='RT', granularity='daily')
    mv_da = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='DA', granularity='daily')

    all_cons = pd.DataFrame({'oops_constraint_num':
        pd.concat([mv_rt['oops_constraint_num'], mv_da['oops_constraint_num']]).unique()})

    det_rt = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='RT')
    det_da = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='DA')
    details = (pd.concat([det_rt, det_da])
               .drop_duplicates('oops_constraint_num')
               [['oops_constraint_num', 'monitored_clean']]
               .rename(columns={'monitored_clean': 'monitored'}))

    da_sum = (mv_da.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('DA_mvalue').reset_index())
    rt_sum = (mv_rt.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('RT_mvalue').reset_index())

    merged = pd.merge(da_sum, rt_sum, on='monitored', how='outer').fillna(0)
    if merged.empty:
        return pd.DataFrame(columns=['monitored', 'DA_mvalue', 'RT_mvalue'])
    merged['monitored'] = merged['monitored'].str.strip()
    # coerce to numeric (column can be object dtype if a side was empty) before rounding
    merged['DA_mvalue'] = pd.to_numeric(merged['DA_mvalue'], errors='coerce').fillna(0).round(0).astype(int)
    merged['RT_mvalue'] = pd.to_numeric(merged['RT_mvalue'], errors='coerce').fillna(0).round(0).astype(int)
    return merged


def _lookup_metadata(monitored_name: str, df: pd.DataFrame) -> dict:
    prior = df[df['monitored'] == monitored_name]
    if prior.empty:
        return {c: '' for c in META_COLS}
    return {c: prior.sort_values('bid_date').iloc[-1].get(c, '') for c in META_COLS}


def _opportunity(monitored_name: str, df: pd.DataFrame, before_dt) -> str:
    prior = df[(df['monitored'] == monitored_name) & (df['bid_date'] < before_dt)]
    if prior.empty:
        return 'new'
    return pd.Timestamp(prior['bid_date'].max()).strftime('%-m/%-d/%Y')


def update_constraints(start_dt: str, end_dt: str, save: bool = True) -> pd.DataFrame:
    df = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'constraints': 'monitored', 'constraints ': 'monitored'})
    df['monitored'] = df['monitored'].str.strip()
    # coerce bad/missing-year bid_dates (e.g. "6/15") to NaT, then drop those rows
    df['bid_date']  = pd.to_datetime(df['bid_date'], format='mixed', errors='coerce')
    df = df.dropna(subset=['bid_date']).reset_index(drop=True)

    date_range = pd.date_range(start=start_dt, end=end_dt, freq='D')
    new_rows   = []

    for dt in date_range:
        dt_str = dt.strftime('%Y-%m-%d')
        print(f"\nFetching {dt_str}...")

        fetched = _fetch_mvalues(dt_str)
        fetched = fetched[
            (fetched['DA_mvalue'] <= THRESHOLD) | (fetched['RT_mvalue'] <= THRESHOLD)
        ].reset_index(drop=True)

        if fetched.empty:
            print(f"  no constraints below threshold")
            continue
        print(f"  {len(fetched)} constraint(s) found")

        existing_mask = df['bid_date'] == dt

        if existing_mask.any():
            for _, frow in fetched.iterrows():
                row_mask = existing_mask & (df['monitored'] == frow['monitored'])
                opp = _opportunity(frow['monitored'], df, before_dt=dt)
                if row_mask.any():
                    df.loc[row_mask, 'DA_mvalue']   = frow['DA_mvalue']
                    df.loc[row_mask, 'RT_mvalue']   = frow['RT_mvalue']
                    df.loc[row_mask, 'opportunity'] = opp
                    print(f"    updated  : {frow['monitored']} (opportunity={opp})")
                else:
                    meta = _lookup_metadata(frow['monitored'], df)
                    new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                     'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                     **meta, "today's wind": '', 'opportunity': opp})
                    print(f"    appended : {frow['monitored']} (new for this date, opportunity={opp})")
        else:
            for _, frow in fetched.iterrows():
                meta = _lookup_metadata(frow['monitored'], df)
                opp  = _opportunity(frow['monitored'], df, before_dt=dt)
                new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                 'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                 **meta, "today's wind": '', 'opportunity': opp})
                print(f"    appended : {frow['monitored']} (opportunity={opp})")

    result = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    result['bid_date'] = pd.to_datetime(result['bid_date'], format='mixed', errors='coerce')
    result = result.dropna(subset=['bid_date']).reset_index(drop=True)

    # Sort: by date first, then within each date by abs(RT - DA) descending
    result['_rank'] = (
        pd.to_numeric(result['RT_mvalue'], errors='coerce').fillna(0) -
        pd.to_numeric(result['DA_mvalue'], errors='coerce').fillna(0)
    ).abs()
    result = (result
              .sort_values(['bid_date', '_rank'], ascending=[True, False])
              .drop(columns='_rank')
              .reset_index(drop=True))

    result['bid_date'] = result['bid_date'].dt.strftime('%-m/%-d/%Y')
    result = result[COL_ORDER]

    print(f"\n{'='*60}")
    print(f"Total rows: {len(result)}  |  New rows added: {len(new_rows)}")
    display(result.tail(len(new_rows) + 3))

    if save:
        result = result.rename(columns={'monitored': 'constraints '})
        result.to_csv(CSV_PATH, index=False)
        print(f"Saved → {CSV_PATH}")

    return result

In [ ]:
today    = pd.Timestamp.now(tz='US/Central').normalize().tz_localize(None)
# today = pd.Timestamp('2026-06-14').normalize().tz_localize(None)
start_dt = (today - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
end_dt   = (today + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

result = update_constraints(start_dt, end_dt, save=True)


Fetching 2026-06-14...
  6 constraint(s) found
    updated  : lnbeaver1-eurk_spa (opportunity=6/13/2026)
    updated  : lndovrt-turcrk2 (opportunity=6/13/2026)
    updated  : lngord-maiz (opportunity=6/13/2026)
    updated  : lnlar3821-sprgfld (opportunity=6/11/2026)
    updated  : lnrussett-sbrown (opportunity=6/13/2026)
    updated  : xfmrduncan-duncan (opportunity=6/13/2026)

Fetching 2026-06-15...
  4 constraint(s) found
    updated  : lnbuln-kel (opportunity=6/13/2026)
    updated  : lnpenn_tap-devilsl (opportunity=new)
    updated  : xfmrduncan-duncan (opportunity=6/14/2026)
    updated  : xfmrsiouxcy-siouxcy (opportunity=6/13/2026)

Fetching 2026-06-16...
  3 constraint(s) found
    appended : lngord-maiz (new for this date, opportunity=6/14/2026)
    appended : lnrussett-sbrown (new for this date, opportunity=6/14/2026)
    appended : xfmrduncan-duncan (new for this date, opportunity=6/15/2026)

Total rows: 252  |  New rows added: 3


,bid_date,monitored,DA_mvalue,RT_mvalue,location,physical_condition,outage_name,start_date,end_date,wind,load,reserve_zone,today's wind,opportunity
246,6/15/2026,lnbuln-kel,-1408.0,-130.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6/13/2026
247,6/15/2026,xfmrduncan-duncan,-1149.0,0.0,"South OKGE, Lawton","high load driven, ng generation up, 64kv","Walters City (OMPA) - Walters Junction 69 kV, +7%",2026-06-10 8:00,2026-06-10 16:00,NaN,NaN,2,NaN,6/14/2026
248,6/16/2026,xfmrduncan-duncan,-2790.0,0.0,"South OKGE, Lawton","high load driven, ng generation up, 64kv","Walters City (OMPA) - Walters Junction 69 kV, +7%",2026-06-10 8:00,2026-06-10 16:00,NaN,NaN,2,,6/15/2026
249,6/16/2026,lngord-maiz,-1448.0,0.0,"Wichita, Kansas","high wind plus small outage, middle wind, all ...","Walters City (OMPA) - Walters Junction 69 kV, ...",2026-05-26 7:32,2026-06-05 15:00,mid,NaN,4,,6/14/2026
250,6/16/2026,lnrussett-sbrown,-1320.0,0.0,South OKGE,"binds high wind, no obvious outage nearby",NaN,NaN,NaN,high,NaN,"3,4",,6/14/2026
251,6/16/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"1: ow: 12, fw: 35, ol: 51, fl: 46\r\n2: ow: 17...",NaN


Saved → /var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv
